# --- Letterboxd 2-user pipeline---

1) Extracts each user's Letterboxd export zip into a separate folder
2) Finds ratings.csv and watched.csv inside (even if nested)
3) Builds a unified user dataframe: ratings + watched-without-rating (neutral rating = 3.0)

In [ ]:
import os
import zipfile
from pathlib import Path
import pandas as pd

In [ ]:
ZIP_USER1 = "/content/letterboxd/letterboxd-celalibr-2026-02-28-20-29-utc.zip"
ZIP_USER2 = "/content/letterboxd/letterboxd-naidaghh-2026-02-28-20-43-utc.zip"

In [ ]:
#Output folders
OUT_BASE = Path("/content/letterboxd_extracted")

OUT_USER1 = OUT_BASE / "user1"
OUT_USER2 = OUT_BASE / "user2"

OUT_USER1.mkdir(parents=True, exist_ok=True)
OUT_USER2.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_zip(zip_path: str, out_dir: Path) -> None:
    """Extract a zip file to out_dir."""
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

In [ ]:
def find_file(root_dir: Path, filename: str) -> Path:
    """
    Find a file by name under root_dir (recursively).
    Returns the first match, raises FileNotFoundError if not found.
    """
    matches = list(root_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root_dir}")
    # If multiple, pick the shortest path (usually the correct one)
    matches = sorted(matches, key=lambda p: len(p.parts))
    return matches[0]

In [ ]:
def build_user_df(extracted_root: Path, neutral_rating: float = 3.0) -> pd.DataFrame:
    """
    Build a user's dataset from Letterboxd export:
    - ratings.csv has explicit ratings
    - watched.csv has watched films (some may not be rated)
    We create a 'film' key using Name + (Year) and fill missing ratings with neutral_rating.
    Returns: DataFrame with columns: film, rating
    """
    ratings_path = find_file(extracted_root, "ratings.csv")
    watched_path = find_file(extracted_root, "watched.csv")

    ratings = pd.read_csv(ratings_path)
    watched = pd.read_csv(watched_path)

    # Standardize columns (Letterboxd exports usually use Name, Year, Rating)
    required_ratings_cols = {"Name", "Year", "Rating"}
    required_watched_cols = {"Name", "Year"}

    if not required_ratings_cols.issubset(set(ratings.columns)):
        raise ValueError(f"ratings.csv columns missing. Found: {ratings.columns.tolist()}")
    if not required_watched_cols.issubset(set(watched.columns)):
        raise ValueError(f"watched.csv columns missing. Found: {watched.columns.tolist()}")

    # Create a stable film identifier: "Title (Year)"
    ratings = ratings.copy()
    watched = watched.copy()
    ratings["film"] = ratings["Name"].astype(str) + " (" + ratings["Year"].astype(int).astype(str) + ")"
    watched["film"] = watched["Name"].astype(str) + " (" + watched["Year"].astype(int).astype(str) + ")"

    # Keep only what we need
    ratings_clean = ratings[["film", "Rating"]].rename(columns={"Rating": "rating"}).dropna()

    # Find watched films that are not rated
    watched_no_rating = watched[~watched["film"].isin(ratings_clean["film"])].copy()
    watched_no_rating["rating"] = float(neutral_rating)
    watched_no_rating = watched_no_rating[["film", "rating"]]

    # Combine
    user_df = pd.concat([ratings_clean, watched_no_rating], ignore_index=True)

    # Ensure numeric rating
    user_df["rating"] = pd.to_numeric(user_df["rating"], errors="coerce")
    user_df = user_df.dropna(subset=["rating"]).reset_index(drop=True)

    return user_df


In [ ]:
#Extract both zips
extract_zip(ZIP_USER1, OUT_USER1)
extract_zip(ZIP_USER2, OUT_USER2)

print("✅ Extracted to:")
print("User1:", OUT_USER1)
print("User2:", OUT_USER2)

✅ Extracted to:
User1: /content/letterboxd_extracted/user1
User2: /content/letterboxd_extracted/user2


In [ ]:
#Build dataframes for each user
user_df_1 = build_user_df(OUT_USER1, neutral_rating=3.0)
user_df_2 = build_user_df(OUT_USER2, neutral_rating=3.0)

print("\n✅ Built user dataframes:")
print("User1 shape:", user_df_1.shape)
print("User2 shape:", user_df_2.shape)



✅ Built user dataframes:
User1 shape: (168, 2)
User2 shape: (123, 2)


In [ ]:
print("User1 head:")
display(user_df_1.head())

User1 head:


,film,rating
0,The Guernsey Literary & Potato Peel Pie Societ...,4.0
1,Top Gun (1986),5.0
2,Scarface (1983),5.0
3,The Phantom of the Opera (2004),4.0
4,Deep Water (2022),0.5


In [ ]:
print("User2 head:")
display(user_df_2.head())

User2 head:


,film,rating
0,Sinners (2025),5.0
1,The Godfather Part II (1974),5.0
2,The Godfather (1972),5.0
3,The Godfather Part III (1990),5.0
4,Normal People (2020),5.0


In [ ]:
import numpy as np
from scipy.stats import pearsonr

In [ ]:
def jaccard_overlap(df1, df2):
    s1, s2 = set(df1["film"]), set(df2["film"])
    if len(s1 | s2) == 0:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

In [ ]:
def rating_corr(df1, df2, min_common=5):
    merged = df1.merge(df2, on="film", suffixes=("_1", "_2"))
    if len(merged) < min_common:
        return 0.0, merged
    corr, _ = pearsonr(merged["rating_1"], merged["rating_2"])
    # clamp negatives to 0 for a "compatibility" interpretation
    return float(max(corr, 0.0)), merged

In [ ]:
def mean_abs_diff_norm(merged):
    if len(merged) == 0:
        return 1.0
    diff = np.mean(np.abs(merged["rating_1"] - merged["rating_2"]))
    return float(diff / 5.0)  # ratings are 0.5..5.0 => normalize by 5

In [ ]:
def compatibility_score(df1, df2):
    overlap = jaccard_overlap(df1, df2)
    corr, merged = rating_corr(df1, df2)
    mad = mean_abs_diff_norm(merged)

    # Weighted score (simple but solid MVP)
    score = 0.35 * overlap + 0.45 * corr + 0.20 * (1 - mad)
    return score, {"overlap": overlap, "corr": corr, "mad_norm": mad, "common_films": len(merged)}

In [ ]:
score, details = compatibility_score(user_df_1, user_df_2)
print("🎬 Common films:", details["common_films"])
print("🔗 Overlap (Jaccard):", round(details["overlap"], 4))
print("📈 Rating agreement (Pearson, clamped):", round(details["corr"], 4))
print("📉 Mean abs diff (normalized):", round(details["mad_norm"], 4))
print("\n✅ Compatibility:", round(score * 100, 2), "%")

🎬 Common films: 48
🔗 Overlap (Jaccard): 0.1975
📈 Rating agreement (Pearson, clamped): 0.3033
📉 Mean abs diff (normalized): 0.1437

✅ Compatibility: 37.69 %


# --- Basic recommendations ---

In [ ]:
!curl -L -o imdb-dataset.zip \
  https://www.kaggle.com/api/v1/datasets/download/ashirwadsangwan/imdb-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1671M  100 1671M    0     0  28.7M      0  0:00:58  0:00:58 --:--:-- 29.2M


In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/letterboxd/imdb-dataset.zip")
out_dir = Path("/content/imdb")

extract_zip(zip_path, out_dir)

In [ ]:
import os
os.listdir("/content/imdb")

['title.akas.tsv',
 'title.ratings.tsv',
 'title.principals.tsv',
 'name.basics.tsv',
 'title.basics.tsv']

In [ ]:
basics = pd.read_csv(
    "/content/imdb/title.basics.tsv",
    sep="\t",
    usecols=[
        "tconst",
        "titleType",
        "primaryTitle",
        "originalTitle",
        "isAdult",
        "startYear",
        "runtimeMinutes",
        "genres"
    ],
    low_memory=False
)

In [ ]:
ratings = pd.read_csv(
    "/content/imdb/title.ratings.tsv",
    sep="\t",
    low_memory=False
)

In [ ]:
print("Basics:", basics.shape)
print("Ratings:", ratings.shape)

Basics: (12315962, 8)
Ratings: (1640037, 3)


In [ ]:
# only movies
basics = basics[basics["titleType"] == "movie"]

# year numeric
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

# ratings numeric
ratings["averageRating"] = pd.to_numeric(ratings["averageRating"], errors="coerce")
ratings["numVotes"] = pd.to_numeric(ratings["numVotes"], errors="coerce")

# merge
imdb = basics.merge(ratings, on="tconst", how="left")

print("IMDb merged:", imdb.shape)

IMDb merged: (738843, 10)


In [ ]:
import re
# keep only movies
imdb = imdb[imdb["titleType"] == "movie"].copy()

# Normalize IMDb titles for matching
def normalize_title(x: str) -> str:
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", "", x)   # remove punctuation
    x = re.sub(r"\s+", " ", x).strip()
    return x

imdb["primary_norm"]  = imdb["primaryTitle"].apply(normalize_title)
imdb["original_norm"] = imdb["originalTitle"].apply(normalize_title)

print("IMDb (movies) shape:", imdb.shape)
imdb.head(3)

IMDb (movies) shape: (738843, 12)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,primary_norm,original_norm
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,45,Romance,5.3,235.0,miss jerry,miss jerry
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,100,"Documentary,News,Sport",5.3,594.0,the corbettfitzsimmons fight,the corbettfitzsimmons fight
2,tt0000502,movie,Bohemios,Bohemios,0,1905.0,100,\N,3.5,26.0,bohemios,bohemios


In [ ]:
def split_title_year(film_key: str):
    m = re.match(r"^(.*)\s+\((\d{4})\)\s*$", str(film_key))
    if not m:
        return str(film_key), np.nan
    return m.group(1), int(m.group(2))

In [ ]:
def prepare_user(df):
    out = df.copy()
    title_year = out["film"].apply(split_title_year)
    out["title"] = title_year.apply(lambda x: x[0])
    out["year"]  = title_year.apply(lambda x: x[1])
    out["title_norm"] = out["title"].apply(normalize_title)
    return out

u1 = prepare_user(user_df_1)   # columns: film, rating, title, year, title_norm
u2 = prepare_user(user_df_2)

In [ ]:
u1.head(3)

,film,rating,title,year,title_norm
0,The Guernsey Literary & Potato Peel Pie Societ...,4.0,The Guernsey Literary & Potato Peel Pie Society,2018,the guernsey literary potato peel pie society
1,Top Gun (1986),5.0,Top Gun,1986,top gun
2,Scarface (1983),5.0,Scarface,1983,scarface


In [ ]:
# --- Build IMDb lookup tables with unique keys to avoid row explosion ---
imdb_primary_lookup = (
    imdb.dropna(subset=["startYear"])
        .sort_values(["numVotes"], ascending=False)  # prefer the most-voted if duplicates
        .drop_duplicates(subset=["primary_norm", "startYear"])
        .set_index(["primary_norm", "startYear"])
)

imdb_original_lookup = (
    imdb.dropna(subset=["startYear"])
        .sort_values(["numVotes"], ascending=False)
        .drop_duplicates(subset=["original_norm", "startYear"])
        .set_index(["original_norm", "startYear"])
)

print("Primary lookup size:", imdb_primary_lookup.shape)
print("Original lookup size:", imdb_original_lookup.shape)

Primary lookup size: (622186, 10)
Original lookup size: (623663, 10)


In [ ]:
# --- Prepare user data (adds title_norm, year) ---
u1 = prepare_user(user_df_1)
u2 = prepare_user(user_df_2)

# --- Primary join ---
u1m = u1.join(
    imdb_primary_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

u2m = u2.join(
    imdb_primary_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

# --- Fallback join (only used to fill missing tconst) ---
u1_fb = u1.join(
    imdb_original_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

u2_fb = u2.join(
    imdb_original_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

# Columns we want to fill from fallback if primary is missing
fill_cols = ["tconst","genres","averageRating","numVotes","runtimeMinutes","primaryTitle","originalTitle","startYear","isAdult","titleType"]

for c in fill_cols:
    if c in u1m.columns and c in u1_fb.columns:
        u1m[c] = u1m[c].combine_first(u1_fb[c])
    if c in u2m.columns and c in u2_fb.columns:
        u2m[c] = u2m[c].combine_first(u2_fb[c])

print("User1 match rate:", round(u1m["tconst"].notna().mean()*100, 2), "%")
print("User2 match rate:", round(u2m["tconst"].notna().mean()*100, 2), "%")

User1 match rate: 73.21 %
User2 match rate: 96.75 %


In [ ]:
!curl -L -o imdb-dataset.zip \
  https://www.kaggle.com/api/v1/datasets/download/ashirwadsangwan/imdb-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1671M  100 1671M    0     0  28.8M      0  0:00:57  0:00:57 --:--:-- 30.4M


In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/letterboxd/imdb-dataset.zip")
out_dir = Path("/content/imdb")

extract_zip(zip_path, out_dir)

In [ ]:
import os
os.listdir("/content/imdb")

['title.akas.tsv',
 'title.ratings.tsv',
 'title.principals.tsv',
 'name.basics.tsv',
 'title.basics.tsv']

In [ ]:
basics = pd.read_csv(
    "/content/imdb/title.basics.tsv",
    sep="\t",
    usecols=[
        "tconst",
        "titleType",
        "primaryTitle",
        "originalTitle",
        "isAdult",
        "startYear",
        "runtimeMinutes",
        "genres"
    ],
    low_memory=False
)

In [ ]:
ratings = pd.read_csv(
    "/content/imdb/title.ratings.tsv",
    sep="\t",
    low_memory=False
)

In [ ]:
print("Basics:", basics.shape)
print("Ratings:", ratings.shape)

Basics: (12315962, 8)
Ratings: (1640037, 3)


In [ ]:
# only movies
basics = basics[basics["titleType"] == "movie"]

# year numeric
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

# ratings numeric
ratings["averageRating"] = pd.to_numeric(ratings["averageRating"], errors="coerce")
ratings["numVotes"] = pd.to_numeric(ratings["numVotes"], errors="coerce")

# merge
imdb = basics.merge(ratings, on="tconst", how="left")

print("IMDb merged:", imdb.shape)

IMDb merged: (738843, 10)


In [ ]:
# keep only movies
imdb = imdb[imdb["titleType"] == "movie"].copy()

# Normalize IMDb titles for matching
def normalize_title(x: str) -> str:
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", "", x)   # remove punctuation
    x = re.sub(r"\s+", " ", x).strip()
    return x

imdb["primary_norm"]  = imdb["primaryTitle"].apply(normalize_title)
imdb["original_norm"] = imdb["originalTitle"].apply(normalize_title)

print("IMDb (movies) shape:", imdb.shape)
imdb.head(3)

IMDb (movies) shape: (738843, 12)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,primary_norm,original_norm
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,45,Romance,5.3,235.0,miss jerry,miss jerry
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,100,"Documentary,News,Sport",5.3,594.0,the corbettfitzsimmons fight,the corbettfitzsimmons fight
2,tt0000502,movie,Bohemios,Bohemios,0,1905.0,100,\N,3.5,26.0,bohemios,bohemios


In [ ]:
def split_title_year(film_key: str):
    m = re.match(r"^(.*)\s+\((\d{4})\)\s*$", str(film_key))
    if not m:
        return str(film_key), np.nan
    return m.group(1), int(m.group(2))

In [ ]:
def prepare_user(df):
    out = df.copy()
    title_year = out["film"].apply(split_title_year)
    out["title"] = title_year.apply(lambda x: x[0])
    out["year"]  = title_year.apply(lambda x: x[1])
    out["title_norm"] = out["title"].apply(normalize_title)
    return out

u1 = prepare_user(user_df_1)   # columns: film, rating, title, year, title_norm
u2 = prepare_user(user_df_2)

In [ ]:
u1.head(3)

,film,rating,title,year,title_norm
0,The Guernsey Literary & Potato Peel Pie Societ...,4.0,The Guernsey Literary & Potato Peel Pie Society,2018,the guernsey literary potato peel pie society
1,Top Gun (1986),5.0,Top Gun,1986,top gun
2,Scarface (1983),5.0,Scarface,1983,scarface


In [ ]:
# --- Build IMDb lookup tables with unique keys to avoid row explosion ---
imdb_primary_lookup = (
    imdb.dropna(subset=["startYear"])
        .sort_values(["numVotes"], ascending=False)  # prefer the most-voted if duplicates
        .drop_duplicates(subset=["primary_norm", "startYear"])
        .set_index(["primary_norm", "startYear"])
)

imdb_original_lookup = (
    imdb.dropna(subset=["startYear"])
        .sort_values(["numVotes"], ascending=False)
        .drop_duplicates(subset=["original_norm", "startYear"])
        .set_index(["original_norm", "startYear"])
)

print("Primary lookup size:", imdb_primary_lookup.shape)
print("Original lookup size:", imdb_original_lookup.shape)

Primary lookup size: (622186, 10)
Original lookup size: (623663, 10)


In [ ]:
# --- Prepare user data (adds title_norm, year) ---
u1 = prepare_user(user_df_1)
u2 = prepare_user(user_df_2)

# --- Primary join ---
u1m = u1.join(
    imdb_primary_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

u2m = u2.join(
    imdb_primary_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

# --- Fallback join (only used to fill missing tconst) ---
u1_fb = u1.join(
    imdb_original_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

u2_fb = u2.join(
    imdb_original_lookup,
    on=["title_norm", "year"],
    rsuffix="_imdb"
)

# Columns we want to fill from fallback if primary is missing
fill_cols = ["tconst","genres","averageRating","numVotes","runtimeMinutes","primaryTitle","originalTitle","startYear","isAdult","titleType"]

for c in fill_cols:
    if c in u1m.columns and c in u1_fb.columns:
        u1m[c] = u1m[c].combine_first(u1_fb[c])
    if c in u2m.columns and c in u2_fb.columns:
        u2m[c] = u2m[c].combine_first(u2_fb[c])

print("User1 match rate:", round(u1m["tconst"].notna().mean()*100, 2), "%")
print("User2 match rate:", round(u2m["tconst"].notna().mean()*100, 2), "%")

User1 match rate: 73.21 %
User2 match rate: 96.75 %


In [ ]:
USER1_DIR = Path("/content/letterboxd_extracted/user1")
USER2_DIR = Path("/content/letterboxd_extracted/user2")

In [ ]:
def find_file(root_dir: Path, filename: str) -> Path:
    matches = list(root_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root_dir}")
    matches = sorted(matches, key=lambda p: len(p.parts))
    return matches[0]

reviews1_path = find_file(USER1_DIR, "reviews.csv")
reviews2_path = find_file(USER2_DIR, "reviews.csv")

reviews1 = pd.read_csv(reviews1_path)
reviews2 = pd.read_csv(reviews2_path)

print("reviews1:", reviews1.shape, "reviews2:", reviews2.shape)
display(reviews1.head(2))

reviews1: (30, 9) reviews2: (7, 9)


,Date,Name,Year,Letterboxd URI,Rating,Rewatch,Review,Tags,Watched Date
0,2025-10-23,Se7en,1995,https://boxd.it/bsgxMT,4.5,NaN,“People no longer pay attention when you tap t...,NaN,2025-09-09
1,2025-11-01,Soul,2020,https://boxd.it/byrEXZ,5.0,Yes,Watching Soul hit me differently. It reminded ...,NaN,2025-10-31


In [ ]:
print("reviews1 cols:", reviews1.columns.tolist())
print("reviews2 cols:", reviews2.columns.tolist())

reviews1 cols: ['Date', 'Name', 'Year', 'Letterboxd URI', 'Rating', 'Rewatch', 'Review', 'Tags', 'Watched Date']
reviews2 cols: ['Date', 'Name', 'Year', 'Letterboxd URI', 'Rating', 'Rewatch', 'Review', 'Tags', 'Watched Date']


In [ ]:
def build_review_df(reviews_df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize reviews to: film, review_text
    Tries common column names found in Letterboxd exports.
    """
    df = reviews_df.copy()

    # Guess title/year columns
    title_col = "Name" if "Name" in df.columns else None
    year_col  = "Year" if "Year" in df.columns else None

    # Guess review text column
    text_candidates = ["Review", "Text", "Body", "Content", "Review Text", "review", "text"]
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if title_col is None or year_col is None or text_col is None:
        raise ValueError(f"Couldn't infer columns. Found: {df.columns.tolist()}")

    df["film"] = df[title_col].astype(str) + " (" + df[year_col].astype(int).astype(str) + ")"
    df["review_text"] = df[text_col].astype(str)

    # Keep only non-empty reviews
    df = df[["film", "review_text"]].copy()
    df["review_text"] = df["review_text"].replace("nan", "").fillna("")
    df = df[df["review_text"].str.strip().str.len() > 0].reset_index(drop=True)

    return df

rev1 = build_review_df(reviews1)
rev2 = build_review_df(reviews2)

print("rev1:", rev1.shape, "rev2:", rev2.shape)
display(rev1.head(2))

rev1: (30, 2) rev2: (7, 2)


,film,review_text
0,Se7en (1995),“People no longer pay attention when you tap t...
1,Soul (2020),Watching Soul hit me differently. It reminded ...


In [ ]:
import string
from collections import Counter

# Simple English stopwords (lightweight; you can expand later)
STOP = set("""
a an the and or but if while of to in on for with as at by from into about
is are was were be been being it this that those these i you he she they we
my your his her their our me him them us
""".split())

def tokenize(text: str):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    toks = [t for t in text.split() if t not in STOP and len(t) >= 3]
    return toks

def top_keywords(review_df: pd.DataFrame, top_k=40):
    all_tokens = []
    for t in review_df["review_text"].astype(str).tolist():
        all_tokens += tokenize(t)
    return Counter(all_tokens).most_common(top_k)

kw1 = top_keywords(rev1, top_k=30)
kw2 = top_keywords(rev2, top_k=30)

print("Top keywords user1:", kw1[:15])
print("Top keywords user2:", kw2[:15])

Top keywords user1: [('like', 12), ('film', 11), ('every', 10), ('how', 9), ('people', 9), ('just', 8), ('life', 8), ('what', 7), ('when', 6), ('time', 6), ('moment', 6), ('even', 6), ('because', 6), ('things', 5), ('matter', 5)]
Top keywords user2: [('💔💔💔', 1), ('great', 1), ('weight', 1), ('growing', 1), ('cool', 1), ('beautifull', 1), ('boş', 1), ('masterpiece', 1), ('storytelling', 1)]


In [ ]:
# Heuristic mapping from keywords to genre boosts (MVP hack, works surprisingly well)
KEYWORD_GENRE_MAP = {
    "funny": ["Comedy"],
    "hilarious": ["Comedy"],
    "laugh": ["Comedy"],
    "dark": ["Thriller", "Horror", "Drama"],
    "scary": ["Horror"],
    "creepy": ["Horror", "Thriller"],
    "romantic": ["Romance"],
    "love": ["Romance", "Drama"],
    "sad": ["Drama"],
    "emotional": ["Drama"],
    "poetic": ["Drama"],
    "slow": ["Drama"],
    "beautiful": ["Drama"],
    "action": ["Action"],
    "violent": ["Action", "Thriller"],
    "mystery": ["Mystery", "Thriller"],
    "suspense": ["Thriller"],
    "epic": ["Adventure", "Action"],
}

def genre_boost_from_keywords(keywords):
    """
    keywords: list of (word, count)
    returns: dict genre -> boost (0..)
    """
    boost = Counter()
    for w, c in keywords:
        if w in KEYWORD_GENRE_MAP:
            for g in KEYWORD_GENRE_MAP[w]:
                boost[g] += c
    # normalize
    total = sum(boost.values()) or 1
    for k in list(boost.keys()):
        boost[k] = boost[k] / total
    return boost

boost1 = genre_boost_from_keywords(kw1)
boost2 = genre_boost_from_keywords(kw2)

print("Boost1:", boost1)
print("Boost2:", boost2)

Boost1: Counter()
Boost2: Counter()


In [ ]:
def genre_profile_from_meta(user_meta: pd.DataFrame, like_threshold=4.0):
    liked = user_meta[user_meta["rating"] >= like_threshold].copy()
    liked = liked.dropna(subset=["genres"])
    g = liked["genres"].astype(str).str.split(",").explode()
    return g.value_counts(normalize=True)

g1 = genre_profile_from_meta(u1m)
g2 = genre_profile_from_meta(u2m)

all_g = sorted(set(g1.index) | set(g2.index))
g_joint = (g1.reindex(all_g).fillna(0) + g2.reindex(all_g).fillna(0)) / 2

# Add review-based boosts (soft)
for genre, b in boost1.items():
    if genre in g_joint.index:
        g_joint.loc[genre] += 0.10 * b   # 10% influence
for genre, b in boost2.items():
    if genre in g_joint.index:
        g_joint.loc[genre] += 0.10 * b

# Re-normalize
g_joint = (g_joint / g_joint.sum()).fillna(0)

print("Joint taste top genres:")
display(g_joint.sort_values(ascending=False).head(12))

Joint taste top genres:


,proportion
genres,
Drama,0.375996
Crime,0.103236
Comedy,0.102874
Romance,0.073533
Biography,0.066288
Adventure,0.044192
Action,0.031997
Animation,0.027047
Horror,0.027047


In [ ]:
seen1 = set(user_df_1["film"])
seen2 = set(user_df_2["film"])
seen_both = seen1 | seen2

# Create imdb film key to compare with Letterboxd keys
imdb_candidates = imdb.copy()
imdb_candidates = imdb_candidates.dropna(subset=["startYear", "primaryTitle", "averageRating", "numVotes", "genres"]).copy()

# Build "film" key for imdb rows: "Title (Year)"
imdb_candidates["film"] = imdb_candidates["primaryTitle"].astype(str) + " (" + imdb_candidates["startYear"].astype(int).astype(str) + ")"

# Basic quality filters (tweak anytime)
imdb_candidates = imdb_candidates[
    (imdb_candidates["numVotes"] >= 5000) &
    (imdb_candidates["averageRating"] >= 6.0) &
    (imdb_candidates["startYear"] >= 1930)
].copy()

# Remove anything already seen by either user
imdb_candidates = imdb_candidates[~imdb_candidates["film"].isin(seen_both)].copy()

print("IMDb candidate pool:", imdb_candidates.shape)
imdb_candidates.head(3)

IMDb candidate pool: (13419, 13)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,primary_norm,original_norm,film
12825,tt0020629,movie,All Quiet on the Western Front,All Quiet on the Western Front,0,1930.0,152,"Drama,War",8.1,71631.0,all quiet on the western front,all quiet on the western front,All Quiet on the Western Front (1930)
12834,tt0020640,movie,Animal Crackers,Animal Crackers,0,1930.0,97,"Comedy,Family,Musical",7.4,16094.0,animal crackers,animal crackers,Animal Crackers (1930)
12870,tt0020691,movie,The Big Trail,The Big Trail,0,1930.0,125,"Adventure,Drama,Romance",7.2,5102.0,the big trail,the big trail,The Big Trail (1930)


In [ ]:
def score_imdb_row(row, joint_genre, min_votes=5000):
    # Genre match
    gs = str(row["genres"]).split(",")
    genre_score = float(joint_genre.reindex(gs).fillna(0).sum())

    # Quality (normalized)
    r = row["averageRating"]
    v = row["numVotes"]
    reliability = 1.0 if v >= min_votes else 0.6
    quality = float(r) / 10.0 * reliability

    # Final score
    return 0.70 * genre_score + 0.30 * quality

imdb_candidates["final_score"] = imdb_candidates.apply(lambda r: score_imdb_row(r, g_joint), axis=1)

top10 = imdb_candidates.sort_values("final_score", ascending=False).head(10)

cols = ["film", "genres", "averageRating", "numVotes", "runtimeMinutes", "final_score"]
display(top10[cols])

,film,genres,averageRating,numVotes,runtimeMinutes,final_score
43432,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,95,0.659474
61493,Jaane Bhi Do Yaaro (1983),"Comedy,Crime,Drama",8.3,17277.0,132,0.656474
49830,Gentlemen of Fortune (1971),"Comedy,Crime,Drama",8.3,14112.0,88,0.656474
551818,Jigarthanda (2014),"Comedy,Crime,Drama",8.2,14662.0,171,0.653474
51417,The Sting (1973),"Comedy,Crime,Drama",8.2,298646.0,129,0.653474
682669,Super Deluxe (2019),"Comedy,Crime,Drama",8.2,19747.0,176,0.653474
207042,Khosla Ka Ghosla! (2006),"Comedy,Crime,Drama",8.2,27041.0,135,0.653474
68821,Time of the Gypsies (1988),"Comedy,Crime,Drama",8.1,34341.0,142,0.650474
194056,Rang De Basanti (2006),"Comedy,Crime,Drama",8.1,126230.0,167,0.650474
631675,"Three Billboards Outside Ebbing, Missouri (2017)","Comedy,Crime,Drama",8.1,601899.0,115,0.650474


In [ ]:
def score_for_user(row, user_genre, min_votes=5000):
    gs = str(row["genres"]).split(",")
    genre_score = float(user_genre.reindex(gs).fillna(0).sum())

    r = row["averageRating"]
    v = row["numVotes"]
    reliability = 1.0 if v >= min_votes else 0.6
    quality = float(r) / 10.0 * reliability

    return 0.75 * genre_score + 0.25 * quality

# Normalize g1,g2 to same index
all_g = sorted(set(g1.index) | set(g2.index))
g1n = (g1.reindex(all_g).fillna(0)); g1n = g1n / (g1n.sum() or 1)
g2n = (g2.reindex(all_g).fillna(0)); g2n = g2n / (g2n.sum() or 1)

# Score per-user and take the minimum (both must like)
imdb_candidates["score_u1"] = imdb_candidates.apply(lambda r: score_for_user(r, g1n), axis=1)
imdb_candidates["score_u2"] = imdb_candidates.apply(lambda r: score_for_user(r, g2n), axis=1)
imdb_candidates["both_like_score"] = np.minimum(imdb_candidates["score_u1"], imdb_candidates["score_u2"])

top10_both = imdb_candidates.sort_values("both_like_score", ascending=False).head(10)
display(top10_both[["film","genres","averageRating","numVotes","both_like_score","score_u1","score_u2"]])

,film,genres,averageRating,numVotes,both_like_score,score_u1,score_u2
43432,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.625842,0.625842,0.667317
61493,Jaane Bhi Do Yaaro (1983),"Comedy,Crime,Drama",8.3,17277.0,0.623342,0.623342,0.664817
49830,Gentlemen of Fortune (1971),"Comedy,Crime,Drama",8.3,14112.0,0.623342,0.623342,0.664817
207042,Khosla Ka Ghosla! (2006),"Comedy,Crime,Drama",8.2,27041.0,0.620842,0.620842,0.662317
682669,Super Deluxe (2019),"Comedy,Crime,Drama",8.2,19747.0,0.620842,0.620842,0.662317
551818,Jigarthanda (2014),"Comedy,Crime,Drama",8.2,14662.0,0.620842,0.620842,0.662317
51417,The Sting (1973),"Comedy,Crime,Drama",8.2,298646.0,0.620842,0.620842,0.662317
70373,Goodfellas (1990),"Biography,Crime,Drama",8.7,1379094.0,0.619939,0.633342,0.619939
68821,Time of the Gypsies (1988),"Comedy,Crime,Drama",8.1,34341.0,0.618342,0.618342,0.659817
194056,Rang De Basanti (2006),"Comedy,Crime,Drama",8.1,126230.0,0.618342,0.618342,0.659817


In [ ]:
display(top10_both)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,primary_norm,original_norm,film,final_score,score_u1,score_u2,both_like_score
43432,tt0059550,movie,Operation 'Y' & Other Shurik's Adventures,Operatsiya 'Y' i drugie priklyucheniya Shurika,0,1965.0,95,"Comedy,Crime,Drama",8.4,16324.0,operation y other shuriks adventures,operatsiya y i drugie priklyucheniya shurika,Operation 'Y' & Other Shurik's Adventures (1965),0.659474,0.625842,0.667317,0.625842
61493,tt0085743,movie,Jaane Bhi Do Yaaro,Jaane Bhi Do Yaaro,0,1983.0,132,"Comedy,Crime,Drama",8.3,17277.0,jaane bhi do yaaro,jaane bhi do yaaro,Jaane Bhi Do Yaaro (1983),0.656474,0.623342,0.664817,0.623342
49830,tt0068519,movie,Gentlemen of Fortune,Dzhentlmeny udachi,0,1971.0,88,"Comedy,Crime,Drama",8.3,14112.0,gentlemen of fortune,dzhentlmeny udachi,Gentlemen of Fortune (1971),0.656474,0.623342,0.664817,0.623342
207042,tt0466460,movie,Khosla Ka Ghosla!,Khosla Ka Ghosla!,0,2006.0,135,"Comedy,Crime,Drama",8.2,27041.0,khosla ka ghosla,khosla ka ghosla,Khosla Ka Ghosla! (2006),0.653474,0.620842,0.662317,0.620842
682669,tt7019942,movie,Super Deluxe,Super Deluxe,0,2019.0,176,"Comedy,Crime,Drama",8.2,19747.0,super deluxe,super deluxe,Super Deluxe (2019),0.653474,0.620842,0.662317,0.620842
551818,tt3569782,movie,Jigarthanda,Jigarthanda,0,2014.0,171,"Comedy,Crime,Drama",8.2,14662.0,jigarthanda,jigarthanda,Jigarthanda (2014),0.653474,0.620842,0.662317,0.620842
51417,tt0070735,movie,The Sting,The Sting,0,1973.0,129,"Comedy,Crime,Drama",8.2,298646.0,the sting,the sting,The Sting (1973),0.653474,0.620842,0.662317,0.620842
70373,tt0099685,movie,Goodfellas,GoodFellas,0,1990.0,145,"Biography,Crime,Drama",8.7,1379094.0,goodfellas,goodfellas,Goodfellas (1990),0.642864,0.633342,0.619939,0.619939
68821,tt0097223,movie,Time of the Gypsies,Dom za vesanje,0,1988.0,142,"Comedy,Crime,Drama",8.1,34341.0,time of the gypsies,dom za vesanje,Time of the Gypsies (1988),0.650474,0.618342,0.659817,0.618342
194056,tt0405508,movie,Rang De Basanti,Rang De Basanti,0,2006.0,167,"Comedy,Crime,Drama",8.1,126230.0,rang de basanti,rang de basanti,Rang De Basanti (2006),0.650474,0.618342,0.659817,0.618342


In [ ]:
display(g_joint.sort_values(ascending=False).head(15))

,proportion
genres,
Drama,0.375996
Crime,0.103236
Comedy,0.102874
Romance,0.073533
Biography,0.066288
Adventure,0.044192
Action,0.031997
Animation,0.027047
Horror,0.027047


In [ ]:
print(kw1[:20])
print(kw2[:20])

[('like', 12), ('film', 11), ('every', 10), ('how', 9), ('people', 9), ('just', 8), ('life', 8), ('what', 7), ('when', 6), ('time', 6), ('moment', 6), ('even', 6), ('because', 6), ('things', 5), ('matter', 5), ('who', 5), ('more', 5), ('can', 5), ('same', 5), ('bir', 5)]
[('💔💔💔', 1), ('great', 1), ('weight', 1), ('growing', 1), ('cool', 1), ('beautifull', 1), ('boş', 1), ('masterpiece', 1), ('storytelling', 1)]


In [ ]:
print("User1:", u1m["tconst"].notna().mean())
print("User2:", u2m["tconst"].notna().mean())

User1: 0.7321428571428571
User2: 0.967479674796748


In [ ]:
top10_both["genres"].value_counts()

,count
genres,
"Comedy,Crime,Drama",9
"Biography,Crime,Drama",1


In [ ]:
import numpy as np

def genre_set(genres_str):
    return set(str(genres_str).split(",")) if pd.notna(genres_str) else set()

def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def mmr_rerank(df, k=10, lambda_=0.75):
    """
    MMR-style re-ranking for diversity.
    lambda_ high -> prioritize relevance (your score)
    lambda_ low  -> prioritize diversity
    """
    df = df.copy()
    df["gset"] = df["genres"].apply(genre_set)

    selected = []
    candidates = df.sort_values("both_like_score", ascending=False).to_dict("records")

    while len(selected) < k and candidates:
        best_idx = None
        best_mmr = -1e9

        for i, cand in enumerate(candidates):
            rel = cand["both_like_score"]  # relevance
            if not selected:
                div_penalty = 0.0
            else:
                # similarity to the most similar selected item
                sims = [jaccard(cand["gset"], s["gset"]) for s in selected]
                div_penalty = max(sims)

            mmr = lambda_ * rel - (1 - lambda_) * div_penalty
            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i

        selected.append(candidates.pop(best_idx))

    out = pd.DataFrame(selected).drop(columns=["gset"])
    return out

# Apply to your big candidate ranking:
top10_diverse = mmr_rerank(imdb_candidates.sort_values("both_like_score", ascending=False), k=10, lambda_=0.78)
display(top10_diverse[["film","genres","averageRating","numVotes","both_like_score"]])

,film,genres,averageRating,numVotes,both_like_score
0,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.625842
1,Champion (2018),"Biography,Drama,Romance",8.2,14195.0,0.576287
2,The Lord of the Rings: The Return of the King ...,"Adventure,Drama,Fantasy",9.0,2147265.0,0.554268
3,Goodfellas (1990),"Biography,Crime,Drama",8.7,1379094.0,0.619939
4,A Brighter Summer Day (1991),"Crime,Drama,Romance",8.2,14832.0,0.613416
5,Mahavatar Narsimha (2024),"Action,Animation,Drama",8.5,45373.0,0.523476
6,Sagara Sangamam (1983),"Drama,Music,Musical",8.8,5874.0,0.512683
7,The Human Condition III: A Soldier's Prayer (1...,"Drama,History,War",8.8,8579.0,0.512683
8,Bride of Frankenstein (1935),"Drama,Horror,Sci-Fi",7.8,58435.0,0.505976
9,Life Is Beautiful (1997),"Comedy,Drama,Romance",8.6,806733.0,0.586287


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def user_review_text(rev_df):
    return " ".join(rev_df["review_text"].astype(str).tolist())

u1_text = user_review_text(rev1)
u2_text = user_review_text(rev2)

# If one user has very little reviews, this will be weak. Still ok as a small bonus.
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
corpus = [u1_text, u2_text]

U = vectorizer.fit_transform(corpus)  # 2 x vocab
u_joint = (U[0] + U[1]) / 2

# Create candidate "text" using title + genres (since we don't have plot)
cand_text = (imdb_candidates["primaryTitle"].fillna("") + " " + imdb_candidates["genres"].fillna("")).tolist()
C = vectorizer.transform(cand_text)

text_sim = cosine_similarity(C, u_joint).reshape(-1)
imdb_candidates["text_sim"] = text_sim

# Blend with both_like_score
imdb_candidates["final_blend"] = 0.85*imdb_candidates["both_like_score"] + 0.15*imdb_candidates["text_sim"]

top10_text = imdb_candidates.sort_values("final_blend", ascending=False).head(10)
display(top10_text[["film","genres","averageRating","numVotes","both_like_score","text_sim","final_blend"]])

,film,genres,averageRating,numVotes,both_like_score,text_sim,final_blend
61533,The King of Comedy (1982),"Comedy,Crime,Drama",7.8,131511.0,0.610842,0.453700,0.587270
49830,Gentlemen of Fortune (1971),"Comedy,Crime,Drama",8.3,14112.0,0.623342,0.326262,0.578780
51417,The Sting (1973),"Comedy,Crime,Drama",8.2,298646.0,0.620842,0.324684,0.576418
68821,Time of the Gypsies (1988),"Comedy,Crime,Drama",8.1,34341.0,0.618342,0.328896,0.574925
38255,The League of Gentlemen (1960),"Comedy,Crime,Drama",7.2,5877.0,0.595842,0.453700,0.574520
102792,The Wounds (1998),"Comedy,Crime,Drama",8.0,12469.0,0.615842,0.324684,0.572168
46361,The Cremator (1969),"Comedy,Crime,Drama",8.0,12661.0,0.615842,0.324684,0.572168
74984,In the Name of the Father (1993),"Biography,Crime,Drama",8.1,201638.0,0.604939,0.368026,0.569402
60759,The Return of Martin Guerre (1982),"Biography,Crime,Drama",7.4,5172.0,0.587439,0.453700,0.567378
682495,The Great Buddha+ (2017),"Comedy,Crime,Drama",7.6,5413.0,0.605842,0.331732,0.564725


In [ ]:
import pandas as pd
valid_regions = ["AZ", "TR", "US", "GB"]
chunks = []

for chunk in pd.read_csv(
    "/content/imdb/title.akas.tsv",
    sep="\t",
    usecols=["titleId","title","region","language","isOriginalTitle"],
    chunksize=200_000
):
    chunk = chunk.rename(columns={"titleId":"tconst"})
    chunk = chunk[chunk["region"].isin(valid_regions)]

    chunk["aka_norm"] = (
        chunk["title"]
        .str.lower()
        .str.strip()
    )

    chunks.append(chunk)

akas = pd.concat(chunks, ignore_index=True)

akas.to_parquet("/content/imdb_akas_clean.parquet")

In [ ]:
akas = pd.read_parquet("/content/imdb_akas_clean.parquet")

In [ ]:
import re

def normalize_title_full(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", "", s)   # remove punctuation/symbols
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Update aka_norm with stronger normalization
akas["aka_norm"] = akas["title"].astype(str).apply(normalize_title_full)

# Also ensure imdb has normalized columns (if not already)
imdb["primary_norm"]  = imdb["primaryTitle"].astype(str).apply(normalize_title_full)
imdb["original_norm"] = imdb["originalTitle"].astype(str).apply(normalize_title_full)

In [ ]:
imdb.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,primary_norm,original_norm
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,45,Romance,5.3,235.0,miss jerry,miss jerry
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,100,"Documentary,News,Sport",5.3,594.0,the corbettfitzsimmons fight,the corbettfitzsimmons fight
2,tt0000502,movie,Bohemios,Bohemios,0,1905.0,100,\N,3.5,26.0,bohemios,bohemios
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,70,"Action,Adventure,Biography",6.0,1054.0,the story of the kelly gang,the story of the kelly gang
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,90,Drama,4.9,38.0,the prodigal son,lenfant prodigue


In [ ]:
# Merge akas with imdb to bring startYear + quality fields
# Keep only needed columns to save RAM
akas2 = akas.merge(
    imdb[[
        "tconst", "startYear", "numVotes", "averageRating",
        "genres", "runtimeMinutes", "primaryTitle", "originalTitle",
        "titleType", "isAdult"
    ]],
    on="tconst",
    how="left"
)

# Remove rows without year (cannot match to "Title (Year)")
akas2 = akas2.dropna(subset=["startYear"]).copy()

# Prefer most voted title when duplicates exist for same (aka_norm, year)
akas2 = akas2.sort_values("numVotes", ascending=False)

aka_lookup = (
    akas2.drop_duplicates(subset=["aka_norm", "startYear"])
         .set_index(["aka_norm", "startYear"])
)

print("aka_lookup size:", aka_lookup.shape)

aka_lookup size: (399664, 13)


In [ ]:
# Ensure u1/u2 title_norm matches the same normalization used for aka_norm
u1 = prepare_user(user_df_1)
u2 = prepare_user(user_df_2)

In [ ]:
u1["title_norm"] = u1["title"].astype(str).apply(normalize_title_full)
u2["title_norm"] = u2["title"].astype(str).apply(normalize_title_full)

In [ ]:
# Join via aka_lookup: (title_norm, year) -> tconst & metadata
u1_aka = u1.join(aka_lookup, on=["title_norm", "year"], rsuffix="_aka")
u2_aka = u2.join(aka_lookup, on=["title_norm", "year"], rsuffix="_aka")

In [ ]:
fill_cols = ["tconst","genres","averageRating","numVotes","runtimeMinutes","primaryTitle","originalTitle","startYear","isAdult","titleType"]

for c in fill_cols:
    if c in u1m.columns and c in u1_aka.columns:
        u1m[c] = u1m[c].combine_first(u1_aka[c])
    if c in u2m.columns and c in u2_aka.columns:
        u2m[c] = u2m[c].combine_first(u2_aka[c])

In [ ]:
print("✅ Match rate after AKAS fallback")
print("User1:", round(u1m["tconst"].notna().mean()*100, 2), "%")
print("User2:", round(u2m["tconst"].notna().mean()*100, 2), "%")

✅ Match rate after AKAS fallback
User1: 75.0 %
User2: 96.75 %


In [ ]:
def genre_profile_from_meta(user_meta: pd.DataFrame, like_threshold=4.0):
    liked = user_meta[user_meta["rating"] >= like_threshold].copy()
    liked = liked.dropna(subset=["genres"])
    g = liked["genres"].astype(str).str.split(",").explode()
    return g.value_counts(normalize=True)

g1 = genre_profile_from_meta(u1m)
g2 = genre_profile_from_meta(u2m)

all_g = sorted(set(g1.index) | set(g2.index))
g1n = g1.reindex(all_g).fillna(0); g1n = g1n / (g1n.sum() or 1)
g2n = g2.reindex(all_g).fillna(0); g2n = g2n / (g2n.sum() or 1)

print("Top genres user1:")
display(g1n.sort_values(ascending=False).head(10))
print("Top genres user2:")
display(g2n.sort_values(ascending=False).head(10))

Top genres user1:


,proportion
genres,
Drama,0.388350
Crime,0.106796
Comedy,0.058252
Biography,0.058252
Romance,0.048544
Adventure,0.048544
Action,0.038835
Animation,0.029126
Horror,0.029126


Top genres user2:


,proportion
genres,
Drama,0.365854
Comedy,0.146341
Crime,0.097561
Romance,0.097561
Biography,0.073171
Adventure,0.048780
Action,0.024390
Animation,0.024390
Fantasy,0.024390


In [ ]:
# Seen by either user
seen_both = set(user_df_1["film"]) | set(user_df_2["film"])

# Build imdb candidate pool
imdb_candidates = imdb.dropna(subset=["startYear","primaryTitle","averageRating","numVotes","genres"]).copy()
imdb_candidates["film"] = imdb_candidates["primaryTitle"].astype(str) + " (" + imdb_candidates["startYear"].astype(int).astype(str) + ")"
imdb_candidates = imdb_candidates[~imdb_candidates["film"].isin(seen_both)].copy()

# Optional quality filters (tweak)
imdb_candidates = imdb_candidates[
    (imdb_candidates["numVotes"] >= 5000) &
    (imdb_candidates["averageRating"] >= 6.5) &
    (imdb_candidates["startYear"] >= 1960)
].copy()

def score_for_user(row, user_genre, min_votes=5000):
    gs = str(row["genres"]).split(",")
    genre_score = float(user_genre.reindex(gs).fillna(0).sum())

    r = float(row["averageRating"])
    v = float(row["numVotes"])
    reliability = 1.0 if v >= min_votes else 0.6
    quality = (r / 10.0) * reliability

    return 0.75 * genre_score + 0.25 * quality

imdb_candidates["score_u1"] = imdb_candidates.apply(lambda r: score_for_user(r, g1n), axis=1)
imdb_candidates["score_u2"] = imdb_candidates.apply(lambda r: score_for_user(r, g2n), axis=1)
imdb_candidates["both_like_score"] = np.minimum(imdb_candidates["score_u1"], imdb_candidates["score_u2"])

# --- MMR rerank for diversity ---
def genre_set(genres_str):
    return set(str(genres_str).split(",")) if pd.notna(genres_str) else set()

def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def mmr_rerank(df, k=10, lambda_=0.78):
    df = df.copy()
    df["gset"] = df["genres"].apply(genre_set)

    selected = []
    candidates = df.sort_values("both_like_score", ascending=False).to_dict("records")

    while len(selected) < k and candidates:
        best_idx = None
        best_mmr = -1e9

        for i, cand in enumerate(candidates):
            rel = cand["both_like_score"]

            if not selected:
                div_penalty = 0.0
            else:
                sims = [jaccard(cand["gset"], s["gset"]) for s in selected]
                div_penalty = max(sims)

            mmr = lambda_ * rel - (1 - lambda_) * div_penalty

            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i

        selected.append(candidates.pop(best_idx))

    return pd.DataFrame(selected).drop(columns=["gset"])

top10_diverse = mmr_rerank(imdb_candidates, k=10, lambda_=0.78)

display(top10_diverse[["film","genres","averageRating","numVotes","both_like_score","score_u1","score_u2"]])

,film,genres,averageRating,numVotes,both_like_score,score_u1,score_u2
0,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.625049,0.625049,0.667317
1,Champion (2018),"Biography,Drama,Romance",8.2,14195.0,0.576359,0.576359,0.607439
2,The Lord of the Rings: The Return of the King ...,"Adventure,Drama,Fantasy",9.0,2147265.0,0.554268,0.567233,0.554268
3,Goodfellas (1990),"Biography,Crime,Drama",8.7,1379094.0,0.619939,0.632549,0.619939
4,A Brighter Summer Day (1991),"Crime,Drama,Romance",8.2,14832.0,0.612767,0.612767,0.625732
5,Mahavatar Narsimha (2024),"Action,Animation,Drama",8.5,45373.0,0.523476,0.554733,0.523476
6,Sagara Sangamam (1983),"Drama,Music,Musical",8.8,5874.0,0.512683,0.525825,0.512683
7,The Human Condition III: A Soldier's Prayer (1...,"Drama,History,War",8.8,8579.0,0.512683,0.533107,0.512683
8,Psycho (1960),"Drama,Horror,Mystery",8.5,780477.0,0.505183,0.547451,0.505183
9,777 Charlie (2022),"Adventure,Comedy,Drama",8.7,47021.0,0.588859,0.588859,0.638232


## ESLINDE ZORDU AMA UPGRADE EDEK

#(A) Exploration vs Exploitation

In [ ]:
# Exploration: give small boost to less obvious films

def exploration_bonus(row):
    # fewer votes -> more novelty
    v = row["numVotes"]
    if v < 20000:
        return 0.05
    elif v < 100000:
        return 0.02
    else:
        return 0.0

imdb_candidates["explore_bonus"] = imdb_candidates.apply(exploration_bonus, axis=1)

imdb_candidates["both_like_score_explore"] = (
    imdb_candidates["both_like_score"] + imdb_candidates["explore_bonus"]
)

In [ ]:
def year_penalty(selected, cand):
    if not selected:
        return 0
    years = [s["startYear"] for s in selected if "startYear" in s]
    if not years:
        return 0
    avg = np.mean(years)
    return abs(cand["startYear"] - avg) / 100  # soft

In [ ]:
#Couple compatibility model
def harmonic(a, b):
    return 2 * a * b / (a + b + 1e-9)

imdb_candidates["couple_score"] = imdb_candidates.apply(
    lambda r: harmonic(r["score_u1"], r["score_u2"]), axis=1
)

In [ ]:
#mood based recoommeder
MOOD_PRESETS = {
    "romantic": {"Romance": 1.4, "Drama": 1.2},
    "fun": {"Comedy": 1.5, "Adventure": 1.2},
    "dark": {"Thriller": 1.4, "Horror": 1.3},
    "epic": {"Action": 1.4, "Fantasy": 1.3}
}

def apply_mood(genre_dist, mood):
    g = genre_dist.copy()
    if mood in MOOD_PRESETS:
        for k, v in MOOD_PRESETS[mood].items():
            if k in g.index:
                g.loc[k] *= v
    return g / g.sum()

In [ ]:
# Choose a mood here:
MOOD = "romantic"   # try: "fun", "dark", "epic", or None

if MOOD:
    g1m = apply_mood(g1n, MOOD)
    g2m = apply_mood(g2n, MOOD)
else:
    g1m, g2m = g1n, g2n

In [ ]:
# Final relevance score (used by ranking/MMR)
imdb_candidates["final_rel"] = imdb_candidates["couple_score"] + imdb_candidates["explore_bonus"]

In [ ]:
def score_for_user(row, user_genre, min_votes=5000):
    gs = str(row["genres"]).split(",")
    genre_score = float(user_genre.reindex(gs).fillna(0).sum())

    r = float(row["averageRating"])
    v = float(row["numVotes"])
    reliability = 1.0 if v >= min_votes else 0.6
    quality = (r / 10.0) * reliability

    return 0.75 * genre_score + 0.25 * quality

# Recompute per-user scores using mood-adjusted genre profiles
imdb_candidates["score_u1_mood"] = imdb_candidates.apply(lambda r: score_for_user(r, g1m), axis=1)
imdb_candidates["score_u2_mood"] = imdb_candidates.apply(lambda r: score_for_user(r, g2m), axis=1)

# Couple score (harmonic mean)
imdb_candidates["couple_score_mood"] = imdb_candidates.apply(
    lambda r: harmonic(r["score_u1_mood"], r["score_u2_mood"]),
    axis=1
)

# Final relevance with exploration
imdb_candidates["final_rel"] = imdb_candidates["couple_score_mood"] + imdb_candidates["explore_bonus"]

In [ ]:
import numpy as np
import pandas as pd

def genre_set(genres_str):
    return set(str(genres_str).split(",")) if pd.notna(genres_str) else set()

def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def year_diversity_bonus(selected, cand):
    if not selected:
        return 0.0
    years = [s["startYear"] for s in selected if "startYear" in s and pd.notna(s["startYear"])]
    if not years:
        return 0.0
    avg = np.mean(years)
    return abs(float(cand["startYear"]) - avg) / 100.0  # soft

def mmr_rerank(df, k=10, lambda_=0.78, rel_col="final_rel"):
    df = df.copy()
    df["gset"] = df["genres"].apply(genre_set)

    selected = []
    candidates = df.sort_values(rel_col, ascending=False).to_dict("records")

    while len(selected) < k and candidates:
        best_idx = None
        best_mmr = -1e9

        for i, cand in enumerate(candidates):
            rel = float(cand.get(rel_col, 0.0))

            if not selected:
                div_penalty = 0.0
            else:
                sims = [jaccard(cand["gset"], s["gset"]) for s in selected]
                div_penalty = max(sims)

            year_bonus = year_diversity_bonus(selected, cand)

            mmr = lambda_ * rel - (1 - lambda_) * div_penalty + 0.05 * year_bonus

            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i

        selected.append(candidates.pop(best_idx))

    out = pd.DataFrame(selected).drop(columns=["gset"])
    return out

In [ ]:
# A) Baseline top10 by both_like_score (old)
top10_baseline = imdb_candidates.sort_values("both_like_score", ascending=False).head(10)

# B) Mood-aware top10 by couple_score_mood (no MMR)
top10_mood = imdb_candidates.sort_values("couple_score_mood", ascending=False).head(10)

# C) Mood-aware + MMR diverse top10
top10_mood_mmr = mmr_rerank(imdb_candidates, k=10, lambda_=0.78, rel_col="final_rel")

print("=== BASELINE ===")
display(top10_baseline[["film","genres","averageRating","numVotes","both_like_score"]])

print(f"=== MOOD ({MOOD}) ===")
display(top10_mood[["film","genres","averageRating","numVotes","couple_score_mood","score_u1_mood","score_u2_mood","explore_bonus"]])

print(f"=== MOOD ({MOOD}) + MMR DIVERSE ===")
display(top10_mood_mmr[["film","genres","averageRating","numVotes","final_rel","couple_score_mood","explore_bonus","score_u1_mood","score_u2_mood"]])

=== BASELINE ===


,film,genres,averageRating,numVotes,both_like_score
43432,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.625049
61493,Jaane Bhi Do Yaaro (1983),"Comedy,Crime,Drama",8.3,17277.0,0.622549
49830,Gentlemen of Fortune (1971),"Comedy,Crime,Drama",8.3,14112.0,0.622549
682669,Super Deluxe (2019),"Comedy,Crime,Drama",8.2,19747.0,0.620049
51417,The Sting (1973),"Comedy,Crime,Drama",8.2,298646.0,0.620049
551818,Jigarthanda (2014),"Comedy,Crime,Drama",8.2,14662.0,0.620049
207042,Khosla Ka Ghosla! (2006),"Comedy,Crime,Drama",8.2,27041.0,0.620049
70373,Goodfellas (1990),"Biography,Crime,Drama",8.7,1379094.0,0.619939
194056,Rang De Basanti (2006),"Comedy,Crime,Drama",8.1,126230.0,0.617549
51247,Paper Moon (1973),"Comedy,Crime,Drama",8.1,57010.0,0.617549


=== MOOD (romantic) ===


,film,genres,averageRating,numVotes,couple_score_mood,score_u1_mood,score_u2_mood,explore_bonus
81240,Life Is Beautiful (1997),"Comedy,Drama,Romance",8.6,806733.0,0.658313,0.619867,0.701842,0.00
711717,Kumbalangi Nights (2019),"Comedy,Drama,Romance",8.5,21265.0,0.655803,0.617367,0.699342,0.02
43432,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.655648,0.641416,0.670526,0.05
63229,The Elusive Summer of '68 (1984),"Comedy,Drama,Romance",8.4,6659.0,0.653293,0.614867,0.696842,0.05
61493,Jaane Bhi Do Yaaro (1983),"Comedy,Crime,Drama",8.3,17277.0,0.653147,0.638916,0.668026,0.05
49830,Gentlemen of Fortune (1971),"Comedy,Crime,Drama",8.3,14112.0,0.653147,0.638916,0.668026,0.05
71781,A Brighter Summer Day (1991),"Crime,Drama,Romance",8.2,14832.0,0.650903,0.643053,0.658947,0.05
52778,Chupke Chupke (1975),"Comedy,Drama,Romance",8.3,13891.0,0.650783,0.612367,0.694342,0.05
38706,The Apartment (1960),"Comedy,Drama,Romance",8.3,217648.0,0.650783,0.612367,0.694342,0.00
706038,Dil Bechara (2020),"Comedy,Drama,Romance",8.3,137377.0,0.650783,0.612367,0.694342,0.00


=== MOOD (romantic) + MMR DIVERSE ===


,film,genres,averageRating,numVotes,final_rel,couple_score_mood,explore_bonus,score_u1_mood,score_u2_mood
0,Operation 'Y' & Other Shurik's Adventures (1965),"Comedy,Crime,Drama",8.4,16324.0,0.705648,0.655648,0.05,0.641416,0.670526
1,Champion (2018),"Biography,Drama,Romance",8.2,14195.0,0.675758,0.625758,0.05,0.609867,0.642500
2,Maheshinte Prathikaaram (2016),"Comedy,Drama,Romance",8.3,11721.0,0.700783,0.650783,0.05,0.612367,0.694342
3,The Truth (1960),"Crime,Drama,Romance",7.6,5406.0,0.685901,0.635901,0.05,0.628053,0.643947
4,Made in Abyss: Dawn of the Deep Soul (2020),"Adventure,Animation,Drama",8.0,7284.0,0.608229,0.558229,0.05,0.571681,0.545395
5,The Legend of Maula Jatt (2022),"Action,Drama,Fantasy",8.4,11624.0,0.603285,0.553285,0.05,0.568407,0.538947
6,The Human Condition III: A Soldier's Prayer (1...,"Drama,History,War",8.8,8579.0,0.595188,0.545188,0.05,0.558496,0.532500
7,Who's Singin' Over There? (1980),"Adventure,Comedy,Drama",8.7,17818.0,0.676594,0.626594,0.05,0.609093,0.645132
8,Lucia (2013),"Drama,Sci-Fi,Thriller",8.3,13852.0,0.588921,0.538921,0.05,0.559270,0.520000
9,Sagara Sangamam (1983),"Drama,Music,Musical",8.8,5874.0,0.592006,0.542006,0.05,0.551858,0.532500


In [ ]:
def genre_combo_diversity(df):
    return df["genres"].nunique()

def year_spread(df):
    return df["startYear"].max() - df["startYear"].min()

print("Baseline genre combos:", genre_combo_diversity(top10_baseline), "year spread:", year_spread(top10_baseline))
print("Mood genre combos:", genre_combo_diversity(top10_mood), "year spread:", year_spread(top10_mood))
print("Mood+MMR genre combos:", genre_combo_diversity(top10_mood_mmr), "year spread:", year_spread(top10_mood_mmr))

Baseline genre combos: 2 year spread: 54.0
Mood genre combos: 3 year spread: 60.0
Mood+MMR genre combos: 10 year spread: 62.0


In [ ]:
# run once in Colab (offline build) to create the minimal imdb_index.parquet
cols = ["tconst","primaryTitle","originalTitle","startYear","runtimeMinutes","genres","averageRating","numVotes","titleType","isAdult"]
imdb_min = imdb[cols].copy()
imdb_min["primary_norm"]  = imdb_min["primaryTitle"].astype(str).apply(normalize_title_full)
imdb_min.to_parquet("/content/imdb_index.parquet", index=False)

In [ ]:
import os

file_path = "/content/imdb_index.parquet"

size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024**2)

print(f"File size: {size_mb:.2f} MB")

File size: 43.22 MB


In [ ]:
from google.colab import files
files.download("/content/imdb_index.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>